# 00 — Data preparation, features and saved splits
Run this notebook first. It supplies the same prepared rows to every model notebook.
See [README](README.md), then [RF](01_Random_Forest.ipynb), [XGBoost](02_XGBoost.ipynb), [GRU](03_GRU.ipynb), [LSTM](04_LSTM.ipynb), and [combined results](05_Combined_Results.ipynb).

This follows the earlier **four-model comparison**, using the attached final notebook's numbered helper-based layout. The two originals are included under `reference/`.

## 1. Setup
Upload this whole extracted project folder to My Drive first. Use the same PROJECT and RUN_DIR in every notebook.

In [ ]:
%pip -q install "scikit-learn==1.6.1" "optuna>=4,<5" "xgboost>=2,<4" joblib

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
sys.path.insert(0, str(PROJECT))
RUN_DIR = PROJECT / "runs" / "real_sessions_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import hashlib, json, uuid, zipfile, shutil
import numpy as np
import pandas as pd
import joblib
import sklearn
from google.colab import files
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from kc_common import *


## 2. Load the real-session ZIP and preserve its split
The archive's `session_manifest.csv` is the authority: we **reuse**, rather than reshuffle, its roughly 60/20/20 groups. Local: 6 train, 2 validation, 2 test. Public validation/test labels are mapped to the corresponding evaluation roles.
Slovenia files are continuous sensor/day segments, not 1,561 independent classroom sessions. This notebook does not turn generated sessions into real evidence.

In [ ]:
uploaded = files.upload()  # Choose KC_CO2_All_Real_Sessions_2026-10-11.zip.
zip_name = next(iter(uploaded))
archive_hash = hashlib.sha256(uploaded[zip_name]).hexdigest()
raw_dir = RUN_DIR / "raw"
if raw_dir.exists():
    shutil.rmtree(raw_dir)  # Clear only this run's previously extracted inputs.
with zipfile.ZipFile(zip_name) as archive:
    assert all(
        not Path(n).is_absolute() and ".." not in Path(n).parts for n in archive.namelist()
    )
    archive.extractall(raw_dir)
manifest_path = next(raw_dir.rglob("session_manifest.csv"))
ROOT = manifest_path.parent
manifest = pd.read_csv(manifest_path)
assert set(manifest.source) == set(
    SOURCE_SHARE
), "Use the real Local, UPC and Slovenia archive."
display(manifest.groupby(["source", "split_role"]).size().rename("Files").reset_index())
manifest.to_csv(RUN_DIR / "split_manifest.csv", index=False)


## 3. Shared feature helper
The next cell shows the actual shared helper used by all models; its source lives in `kc_common.py` so recursive forecasts use exactly the same definition.
Inputs include the current readings, lagged CO₂, slopes, rolling statistics, elapsed time, occupancy when available, room, and hour. CO₂ and predicted change use ppm; slopes use ppm/minute.

In [ ]:
import inspect

print(inspect.getsource(make_features))
print("Numeric features:", NUM)


## 4. Clean and build each horizon
Process each file and room/day separately. Local and UPC readings become one-minute last observations. Rows missing CO₂, temperature or humidity are dropped; gaps break sequences. Slovenia uses weekday 08:00–16:00 forecast origins, as before.
Every origin needs 10 minutes of history and 10 minutes of future data, even for the five-minute model. This keeps both horizons on the same origin rows. No extra spike filtering is added.
The recurrent sequence is 11 one-minute observations of CO₂, temperature and humidity, plus the engineered static features.

In [ ]:
def prepare_horizon(horizon):
    from numpy.lib.stride_tricks import sliding_window_view

    frames = []
    seq = []
    contexts = []
    for row in manifest.itertuples(index=False):
        g = pd.read_csv(ROOT / row.file, parse_dates=["time"])
        if g.empty:
            continue
        g["day_key"] = g.time.dt.strftime("%Y-%m-%d")
        for (_, room), piece in g.groupby(["day_key", "room"], sort=True):
            if row.source == "Slovenia":
                base = piece[["time", "co2", "temp", "humidity", "people"]].copy()
            else:
                base = (
                    piece.set_index("time")[["co2", "temp", "humidity", "people"]]
                    .resample("1min", closed="right", label="right")
                    .last()
                    .reset_index()
                )
            base = base.dropna(subset=["co2", "temp", "humidity"]).sort_values("time")
            seg = base.time.diff().ne(pd.Timedelta(minutes=1)).cumsum()
            for _, s in base.groupby(seg, sort=False):
                n = len(s)
                if n < 21:
                    continue
                s = s.reset_index(drop=True)
                pos = np.arange(10, n - 10)
                if row.source == "Slovenia":
                    stamp = s.time.iloc[pos]
                    pos = pos[
                        (
                            (stamp.dt.dayofweek < 5)
                            & (stamp.dt.hour >= 8)
                            & (stamp.dt.hour < 16)
                        ).to_numpy()
                    ]
                if not len(pos):
                    continue
                c = sliding_window_view(s.co2.to_numpy(dtype=np.float32), 11)[pos - 10].copy()
                t = sliding_window_view(s.temp.to_numpy(dtype=np.float32), 11)[pos - 10].copy()
                h = sliding_window_view(s.humidity.to_numpy(dtype=np.float32), 11)[
                    pos - 10
                ].copy()
                origin = s.iloc[pos]
                initial = np.full(len(pos), s.co2.iloc[0])
                elapsed = pos.astype(float)
                people = origin.people.to_numpy(dtype=float)
                times = origin.time
                f = make_features(
                    c, t, h, initial, elapsed, people, times, np.repeat(room, len(pos))
                )
                f["source"] = row.source
                f["role"] = row.split_role
                f["session"] = row.file
                f["time"] = times.to_numpy()
                f["actual_future"] = s.co2.to_numpy()[pos + horizon]
                f["target_direct"] = f.actual_future - f.CO2_now
                f["target_recursive"] = s.co2.to_numpy()[pos + 1] - f.CO2_now
                frames.append(f)
                seq.append(np.stack([c, t, h], axis=-1))
                contexts.append((c, t, h, initial, elapsed))
    data = pd.concat(frames, ignore_index=True)
    sequence = np.concatenate(seq).astype(np.float32)
    history = {
        key: np.concatenate([item[k] for item in contexts])
        for k, key in enumerate(["c", "t", "h", "initial", "elapsed"])
    }
    assert len(data) == len(sequence) == len(history["c"])
    assert data.groupby(["source", "session"]).role.nunique().eq(1).all()
    return (data, sequence, history)


In [ ]:
DATA, SEQ, HIST = {}, {}, {}
for horizon in (5, 10):
    DATA[horizon], SEQ[horizon], HIST[horizon] = prepare_horizon(horizon)
    display(
        DATA[horizon]
        .groupby(["source", "role"])
        .size()
        .rename(f"Rows at {horizon} min")
        .reset_index()
    )


## 5. Fit preprocessing on training rows only
Missing numeric features use training medians. Room uses one-hot encoding. Neural features and sequences are scaled using training data only.
The earlier search caps are retained: at most 12,000 Slovenia training rows and 4,000 validation rows per public source per trial. Final fits use **all training rows**, not training plus validation. Training losses and trial validation MAE use the source weights shown below.

In [ ]:
print("Source weights:", SOURCE_SHARE)
print("Trial training caps:", TRIAL_TRAIN_CAP)
print("Trial validation caps:", TRIAL_VAL_CAP)


In [ ]:
STATE = {}
for horizon in (5, 10):
    d = DATA[horizon]
    group = role_group(d.role)
    assert (group != "Invalid").all()
    assert d.groupby(["source", "session"]).role.nunique().eq(1).all()
    state = {
        "train": np.flatnonzero(group == "Training"),
        "val": np.flatnonzero(group == "Validation"),
        "test": np.flatnonzero(group == "Test"),
    }
    tr = d.iloc[state["train"]]
    va = d.iloc[state["val"]]
    state["trial_train"] = sample_per_source(tr, TRIAL_TRAIN_CAP, SEED + horizon)
    state["trial_val"] = sample_per_source(va, TRIAL_VAL_CAP, SEED + 2 * horizon)
    state["prep"] = ColumnTransformer(
        [
            ("numeric", SimpleImputer(strategy="median", keep_empty_features=True), NUM),
            ("room", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["room"]),
        ]
    )
    state["X"] = state["prep"].fit(tr[FEATURES]).transform(d[FEATURES]).astype(np.float32)
    state["static_scaler"] = StandardScaler().fit(state["X"][state["train"]])
    state["Xnn"] = state["static_scaler"].transform(state["X"]).astype(np.float32)
    mu = SEQ[horizon][state["train"]].reshape(-1, 3).mean(axis=0)
    sd = SEQ[horizon][state["train"]].reshape(-1, 3).std(axis=0).clip(0.001)
    state["Snn"] = ((SEQ[horizon] - mu) / sd).astype(np.float32)
    STATE[horizon] = state
    print(
        horizon,
        "trial train",
        len(state["trial_train"]),
        "trial validation",
        len(state["trial_val"]),
        "final train",
        len(state["train"]),
        "full validation",
        len(state["val"]),
        "test",
        len(state["test"]),
    )


## 6. Save the handoff and persistence baseline
All later notebooks load `prepared.joblib`; they do not repeat cleaning or choose new splits. Rerunning this notebook creates a new run ID: rerun all models afterward, or change RUN_DIR for a separate experiment.
Only load the joblib files you generated yourself.

In [ ]:
run_id = uuid.uuid4().hex
bundle = {"DATA": DATA, "STATE": STATE, "HIST": HIST, "SEQ": SEQ, "run_id": run_id}
joblib.dump(bundle, RUN_DIR / "prepared.joblib", compress=3)
metadata = {
    "run_id": run_id,
    "archive_sha256": archive_hash,
    "source_weights": SOURCE_SHARE,
    "seed": SEED,
    "trials": N_TRIALS,
    "sklearn_version": sklearn.__version__,
    "horizons": [5, 10],
}
(RUN_DIR / "preparation.json").write_text(json.dumps(metadata, indent=2))
rows = []
for horizon in (5, 10):
    for split, key in [("Validation", "val"), ("Test", "test")]:
        ids = STATE[horizon][key]
        d = DATA[horizon].iloc[ids]
        p = d[["source", "session", "time", "actual_future", "CO2_now"]].copy()
        p["row_id"] = ids
        p["predicted_future"] = p.CO2_now
        p["Model"] = "Persistence"
        p["Horizon_min"] = horizon
        p["Approach"] = "baseline"
        p["Split"] = split
        p["run_id"] = run_id
        rows.append(p)
pd.concat(rows).to_csv(RUN_DIR / "persistence_predictions.csv.gz", index=False)
print("Prepared data saved. Run notebooks 01–04 next, in any order.")
